# 🔤 Stock Recommender — Step 3: NLP Preprocessing

**Goal:** Transform raw titles into clean, feature-rich text representations
that FinBERT and the downstream classifier can use effectively.

**What this step produces:**
- `df_clean` — deduplicated, crypto-dropped, text-cleaned dataframe
- `tokens` column — lemmatized token lists for each article
- `wordclouds` — per-region and per-label visualisations
- `30+ keyword lexicon features` — bullish/bearish/event/analyst signals
- `step3_nlp.csv` — saved output for Step 4 (FinBERT inference)

**Dataset entering this step:**
```
11,967 rows → drop 484 crypto → 11,483 equity articles
45 tickers  |  US + India only  |  imbalance 1.48x
```

```
Step 1 → EDA & Audit            ✅ DONE
Step 2 → Price Labels           ✅ DONE (11,483 labeled rows)
Step 3 → NLP Preprocessing      ← YOU ARE HERE
Step 4 → FinBERT Features       ← next
Step 5 → Model Training         ← after Step 4
Step 6 → Evaluation & Tune      ← after Step 5
```


## ⚙️ 0. Install & Imports

In [ ]:
import subprocess, sys
for pkg in ["pandas","numpy","matplotlib","seaborn","nltk","wordcloud","scikit-learn"]:
    subprocess.run([sys.executable,"-m","pip","install",pkg,"-q"], check=False)

import re, warnings
from collections import Counter
from IPython.display import display

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from wordcloud import WordCloud

import nltk
for resource in ["punkt","stopwords","wordnet","averaged_perceptron_tagger",
                 "punkt_tab","omw-1.4"]:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 110)
plt.rcParams["figure.dpi"] = 120
sns.set_style("whitegrid")

LABEL_MAP  = {0:"SELL", 1:"HOLD", 2:"BUY"}
LABEL_PAL  = {0:"#F44336", 1:"#9E9E9E", 2:"#4CAF50"}
REGION_PAL = {"US":"#4CAF50", "IN":"#2196F3"}

print("✅ Imports ready.")


## 📂 1. Load Step 2 Output & Drop Crypto

In [ ]:
df = pd.read_csv("step2_labeled.csv")
df['pub_dt']   = pd.to_datetime(df['published_at'], utc=True, errors='coerce')
df['pub_date'] = pd.to_datetime(df['pub_date'])

print(f"Loaded     : {len(df):,} rows")

# Drop crypto — decision made in Step 2 review
df = df[df['region'] != 'CRYPTO'].copy()
df.reset_index(drop=True, inplace=True)

print(f"After drop : {len(df):,} rows  ({len(df[df['region']=='US']):,} US "
      f"| {len(df[df['region']=='IN']):,} IN)")
print(f"Tickers    : {df['ticker'].nunique()}")
print()

# Confirm label distribution after crypto drop
counts = df['label'].value_counts().sort_index()
total  = len(df)
print("Label distribution (equity-only):")
for lv,cnt in counts.items():
    bar = '█' * (cnt//100)
    print(f"  {LABEL_MAP[lv]:4s}: {cnt:5,}  ({cnt/total*100:5.1f}%)  {bar}")
print(f"  Imbalance ratio: {counts.max()/counts.min():.2f}x")


## 🔬 2. Raw Text Inspection

In [ ]:
# Inspect raw titles before any cleaning — know what we're dealing with
print("SAMPLE RAW TITLES WITH LABELS")
print("="*80)
for label in [0,1,2]:
    print(f"\n── {LABEL_MAP[label]} ──")
    sample = df[df['label']==label]['title'].sample(5, random_state=42)
    for t in sample:
        print(f"  • {t}")

print()
print("COMMON NOISE PATTERNS TO CLEAN:")
noise_patterns = {
    "Source attribution at end"  : df['title'].str.contains(r'\s[-–|]\s+\w', regex=True).sum(),
    "Percentage mentions"        : df['title'].str.contains(r'\d+\.?\d*\s*%', regex=True).sum(),
    "Currency symbols (₹,$,€)"  : df['title'].str.contains(r'[₹$€£]', regex=True).sum(),
    "Large number abbrev (B,M,K)": df['title'].str.contains(r'\d+[BMKbmk]', regex=True).sum(),
    "URLs present"               : df['title'].str.contains(r'https?://', regex=True).sum(),
    "Titles > 120 chars"         : (df['title'].str.len() > 120).sum(),
    "All-caps words"             : df['title'].str.contains(r'[A-Z]{4,}', regex=True).sum(),
}
for pattern, count in noise_patterns.items():
    print(f"  {pattern:<38}: {count:,}  ({count/len(df)*100:.1f}%)")


## 🧹 3. Multi-Stage Regex Cleaning

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Cleaning pipeline — each stage targets a specific noise type.
# Order matters: normalise numbers BEFORE removing special chars.
# ─────────────────────────────────────────────────────────────────────────────

# Source attributions to strip from title endings
SOURCE_PATTERN = re.compile(
    r'\s*[-–|]\s*('
    r"Barron\'s|Bloomberg|Reuters|CNBC|Forbes|Motley Fool|Seeking Alpha|"
    r'Yahoo Finance|MarketWatch|The Hindu|Economic Times|Moneycontrol|'
    r'Mint|LiveMint|Business Standard|Financial Express|Investing\.com|'
    r'Trefis|simplywall\.st|Euronews|IBD|TechCrunch|Business Insider|'
    r'Wall Street Journal|WSJ|FT|Financial Times|ZDNet|TheStreet|'
    r'Benzinga|Zacks|Nasdaq\.com|fool\.com|NDTV|ETMarkets|'
    r'ET Now|ANI|PTI|Hindu BusinessLine|scanx\.trade'
    r').*$',
    re.IGNORECASE
)

# Extreme move flag — capture BEFORE cleaning wipes the numbers
EXTREME_MOVE_RE = re.compile(
    r'((?:surges?|jumps?|soars?|plunges?|crashes?|drops?|falls?|tumbles?)'
    r'\s+(?:over|more than|nearly|almost)?\s*'
    r'(\d{1,2}(?:\.\d)?\s*(?:percent|%)))',
    re.IGNORECASE
)

def extract_extreme_move(text: str) -> tuple:
    match = EXTREME_MOVE_RE.search(str(text))
    if match:
        try:
            pct_str = re.search(r'\d+\.?\d*', match.group(2)).group()
            return 1, float(pct_str)
        except:
            return 1, 0.0
    return 0, 0.0

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ""

    # Stage 1 — strip source attribution suffix
    text = SOURCE_PATTERN.sub('', text)

    # Stage 2 — normalise percentages  "12.5%" → "12.5 percent"
    text = re.sub(r'(\d+\.?\d*)\s*%', r'\1 percent', text)

    # Stage 3 — normalise large numbers  "1.2B" → "1.2 billion"
    text = re.sub(r'(\d+\.?\d*)\s*[Bb](?=\s|$|\b)', r'\1 billion', text)
    text = re.sub(r'(\d+\.?\d*)\s*[Mm](?=\s|$|\b)', r'\1 million', text)
    text = re.sub(r'(\d+\.?\d*)\s*[Kk](?=\s|$|\b)', r'\1 thousand', text)

    # Stage 4 — currency symbols
    text = re.sub(r'[₹$€£¥]', ' ', text)

    # Stage 5 — strip URLs
    text = re.sub(r'https?\S+', '', text)

    # Stage 6 — remove non-alphanumeric except spaces and basic punctuation
    text = re.sub(r"[^a-zA-Z0-9\s.,!?'\-]", ' ', text)

    # Stage 7 — collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply
df['title_clean'] = df['title'].apply(clean_text)

# Extract extreme move flag BEFORE cleaning removes numbers
df[['flag_extreme_move', 'extreme_move_pct']] = df['title'].apply(
    lambda t: pd.Series(extract_extreme_move(t))
)

# Before/after comparison
print("CLEANING EXAMPLES (before → after):")
print("="*80)
sample_idx = df[df['title'].str.len() > 60].sample(8, random_state=7).index
for idx in sample_idx:
    orig  = df.loc[idx,'title']
    clean = df.loc[idx,'title_clean']
    if orig != clean:
        print(f"  BEFORE: {orig[:95]}")
        print(f"  AFTER : {clean[:95]}")
        print()

print(f"Extreme move flags: {df['flag_extreme_move'].sum():,} articles "
      f"({df['flag_extreme_move'].mean()*100:.1f}%)")
print(f"Avg extreme move % : {df[df['flag_extreme_move']==1]['extreme_move_pct'].mean():.1f}%")


## ✂️ 4. Tokenization & Lemmatization

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Custom stopword list — standard NLTK list + financial filler words
# that carry no signal (stock, share, company, market, etc.)
# ─────────────────────────────────────────────────────────────────────────────

BASE_STOPS = set(stopwords.words('english'))
FINANCIAL_STOPS = {
    'stock','stocks','share','shares','company','companies','market',
    'markets','news','google','says','say','said','report','reports',
    'reported','get','got','could','would','may','might','also','new',
    'year','years','one','two','three','like','quarter','quarterly',
    'week','month','day','today','monday','tuesday','wednesday',
    'thursday','friday','january','february','march','april','may',
    'june','july','august','september','october','november','december',
    'ltd','inc','corp','co','group','limited'
}
ALL_STOPS = BASE_STOPS | FINANCIAL_STOPS

lemmatizer = WordNetLemmatizer()

def tokenize(text: str) -> list:
    tokens = word_tokenize(text.lower())
    tokens = [
        lemmatizer.lemmatize(t)
        for t in tokens
        if t.isalpha()
        and t not in ALL_STOPS
        and len(t) > 2
    ]
    return tokens

print("Tokenizing titles...")
df['tokens']      = df['title_clean'].apply(tokenize)
df['token_count'] = df['tokens'].apply(len)

print(f"✅ Done. Token count stats:")
print(df['token_count'].describe().round(2).to_string())

print()
print("SAMPLE TOKENIZED TITLES:")
for _, row in df.sample(5, random_state=42).iterrows():
    print(f"  [{LABEL_MAP[row['label']]}] {row['title_clean'][:70]}")
    print(f"        → {row['tokens'][:12]}")
    print()


## 📊 5. Vocabulary & Token Distribution Analysis

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Analyse vocabulary per label — do BUY articles use distinct words from SELL?
# This validates that NLP features will carry discriminative signal.
# ─────────────────────────────────────────────────────────────────────────────

vocab_by_label = {}
for lv in [0,1,2]:
    all_tokens = [t for tokens in df[df['label']==lv]['tokens'] for t in tokens]
    vocab_by_label[lv] = Counter(all_tokens)

# Top 20 per label
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, lv in zip(axes, [0,1,2]):
    top = vocab_by_label[lv].most_common(20)
    words, counts = zip(*top)
    bars = ax.barh(words[::-1], counts[::-1],
                   color=LABEL_PAL[lv], alpha=0.85, edgecolor='white')
    n_art = len(df[df['label']==lv])
    ax.set_title(f"Top 20 Tokens - {LABEL_MAP[lv]} (n={n_art:,} articles)",
                 fontweight='bold', fontsize=12)
    ax.set_xlabel("Frequency")
    for bar in bars:
        ax.text(bar.get_width()+5, bar.get_y()+bar.get_height()/2,
                str(int(bar.get_width())), va='center', fontsize=8)

plt.suptitle("Most Frequent Tokens by Label Class", fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("step3_top_tokens.png", bbox_inches='tight', dpi=150)
plt.show()


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Distinctive tokens — words that are MUCH more common in one label vs others
# Computed as: count_in_label / count_in_all_other_labels (lift ratio)
# ─────────────────────────────────────────────────────────────────────────────

total_vocab = Counter()
for lv in [0,1,2]:
    total_vocab.update(vocab_by_label[lv])

def lift_words(label_counter, all_counter, min_count=20, top_n=15):
    lifts = {}
    for word, cnt in label_counter.items():
        if cnt < min_count:
            continue
        other_cnt = all_counter[word] - cnt + 1   # +1 smoothing
        lifts[word] = cnt / other_cnt
    return sorted(lifts.items(), key=lambda x: -x[1])[:top_n]

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
for ax, lv in zip(axes, [0,1,2]):
    distinctive = lift_words(vocab_by_label[lv], total_vocab)
    if not distinctive:
        continue
    words, lifts = zip(*distinctive)
    bars = ax.barh(words[::-1], lifts[::-1],
                   color=LABEL_PAL[lv], alpha=0.85, edgecolor='white')
    ax.set_title(f"Most Distinctive Tokens - {LABEL_MAP[lv]} (lift ratio vs other classes)",
                 fontweight='bold', fontsize=12)
    ax.set_xlabel("Lift Ratio (higher = more distinctive)")
    ax.axvline(1.0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)

plt.suptitle("Distinctive Token Lift Analysis (tokens disproportionate per label)",
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig("step3_lift_tokens.png", bbox_inches='tight', dpi=150)
plt.show()

print("High-lift SELL tokens should include: downgrade, miss, decline, cut, layoff")
print("High-lift BUY tokens should include: upgrade, beat, surge, record, raised")
print("These validate our lexicon design in the next cell.")


## ☁️ 6. Word Clouds

In [ ]:
def make_wc(tokens_series, colormap='viridis', max_words=120, bg='white'):
    text = ' '.join(
        word for tokens in tokens_series for word in tokens
    )
    return WordCloud(
        width=800, height=380,
        background_color=bg,
        colormap=colormap,
        max_words=max_words,
        collocations=False,
        stopwords=set()    # already cleaned
    ).generate(text)

fig, axes = plt.subplots(2, 3, figsize=(22, 10))

# Row 1: by label
configs_label = [
    (df[df['label']==0], 'Reds',   f"SELL Articles (n={( df['label']==0).sum():,})"),
    (df[df['label']==1], 'Greys',  f"HOLD Articles (n={(df['label']==1).sum():,})"),
    (df[df['label']==2], 'Greens', f"BUY Articles  (n={(df['label']==2).sum():,})"),
]
for ax, (subset, cmap, title) in zip(axes[0], configs_label):
    wc = make_wc(subset['tokens'], colormap=cmap)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.axis('off')

# Row 2: by region (US, IN) + overall high-confidence
configs_region = [
    (df[df['region']=='US'],              'Blues',   f"US Market (n={( df['region']=='US').sum():,})"),
    (df[df['region']=='IN'],              'Oranges', f"India Market (n={(df['region']=='IN').sum():,})"),
    (df[df['horizons_agree']==True],      'Purples', f"High-Confidence Labels (n={(df['horizons_agree']==True).sum():,})"),
]
for ax, (subset, cmap, title) in zip(axes[1], configs_region):
    wc = make_wc(subset['tokens'], colormap=cmap)
    ax.imshow(wc, interpolation='bilinear')
    ax.set_title(title, fontweight='bold', fontsize=12)
    ax.axis('off')

plt.suptitle("Word Clouds — By Label & Market Region", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("step3_wordclouds.png", bbox_inches='tight', dpi=150)
plt.show()


## 🔑 7. Keyword Lexicon Features

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Lexicons are built from three sources:
#   1. Financial NLP literature (Loughran-McDonald word list)
#   2. Empirical validation from Step 3 lift analysis
#   3. Domain knowledge for Indian market terminology
# ─────────────────────────────────────────────────────────────────────────────

STRONG_BULL = {
    'upgrade','upgraded','outperform','overweight','strong buy','beat','beats',
    'surpass','surge','soar','jump','rally','breakout','breakthrough','raised',
    'raise','record high','acquisition','approve','approved','fda','launch',
    'rebound','recover','turnaround','revenue growth','expansion','profit',
    'dividend','buyback','partnership','win','award','accelerate',
    'upside','momentum','bullish','positive surprise','guidance raise',
    'earnings beat','top line beat','margin expansion','market share gain',
}
WEAK_BULL = {
    'gain','rise','rose','grew','grow','higher','improve','boost','solid',
    'strong','better','increase','optimistic','potential','opportunity',
    'positive','stabilise','stabilize','recovery','outperform',
}
STRONG_BEAR = {
    'downgrade','downgraded','underperform','underweight','sell','miss',
    'misses','missed','disappoint','plunge','crash','tumble','tank',
    'collapse','cut','reduce','decrease','layoff','layoffs','bankrupt',
    'bankruptcy','fine','lawsuit','fraud','scandal','investigation','sec',
    'recall','default','write-off','writedown','loss','losses','bearish',
    'short','overvalued','margin pressure','guidance cut','earnings miss',
    'revenue miss','downside','headwind','concern','risk','warning',
    'regulatory','probe','allegation','resign','resign',
}
WEAK_BEAR = {
    'fall','fell','lower','weak','below','pressure','challenge','difficult',
    'uncertain','volatility','worry','caution','slowdown','decline',
    'negative','disappoint',
}
NEUTRAL_SIGNALS = {
    'hold','neutral','mixed','flat','unchanged','stable','steady',
    'maintain','reiterate','in-line','inline','meet','meets',
}
EVENT_SIGNALS = {
    'earnings','results','quarterly','q1','q2','q3','q4','annual',
    'guidance','forecast','outlook','target','estimate','revenue',
    'merger','acquisition','ipo','spinoff','split','dividend',
    'rate','inflation','gdp','macro','regulatory','policy','budget',
    'capex','margins','ebitda','eps','pat','net profit',
}
ANALYST_SIGNALS = {
    'analyst','analysts','target','rating','upgrade','downgrade',
    'initiate','coverage','reiterate','consensus','wall street',
    'price target','buy rating','sell rating','hold rating','broker',
    'brokerage','morgan stanley','goldman','jpmorgan','citi','ubs',
    'jefferies','bernstein','macquarie','nomura','kotak','motilal',
    'iifl','geojit','sharekhan','icici securities',
}
# Indian market specific
INDIA_BULL = {
    'nifty','sensex','fii buying','dii buying','operator','promoter buying',
    'bulk deal','block deal','upper circuit','52 week high','all time high',
    'sebi approved','rbi approved','nclt approved','nse listed',
}
INDIA_BEAR = {
    'lower circuit','52 week low','fii selling','promoter pledge',
    'sebi probe','sebi notice','income tax','ed raid','npa','bad loan',
    'stressed asset',
}

def count_kw(text: str, kw_set: set) -> int:
    t = text.lower()
    return sum(1 for kw in kw_set if kw in t)

def has_price_target(text: str) -> int:
    return int(bool(re.search(
        r'(price target|pt of|target of|target price|tp:|₹\s*\d|\$\s*\d)',
        text, re.IGNORECASE
    )))

def extract_pct_magnitude(text: str) -> float:
    matches = re.findall(r'([+-]?\d+\.?\d*)\s*(?:percent|%)', text, re.IGNORECASE)
    return float(matches[0]) if matches else 0.0

print("Computing lexicon features for 11,483 articles...")

df['kw_strong_bull']  = df['title_clean'].apply(lambda t: count_kw(t, STRONG_BULL))
df['kw_weak_bull']    = df['title_clean'].apply(lambda t: count_kw(t, WEAK_BULL))
df['kw_strong_bear']  = df['title_clean'].apply(lambda t: count_kw(t, STRONG_BEAR))
df['kw_weak_bear']    = df['title_clean'].apply(lambda t: count_kw(t, WEAK_BEAR))
df['kw_neutral']      = df['title_clean'].apply(lambda t: count_kw(t, NEUTRAL_SIGNALS))
df['kw_event']        = df['title_clean'].apply(lambda t: count_kw(t, EVENT_SIGNALS))
df['kw_analyst']      = df['title_clean'].apply(lambda t: count_kw(t, ANALYST_SIGNALS))
df['kw_india_bull']   = df['title_clean'].apply(lambda t: count_kw(t, INDIA_BULL))
df['kw_india_bear']   = df['title_clean'].apply(lambda t: count_kw(t, INDIA_BEAR))
df['has_price_target']= df['title_clean'].apply(has_price_target)
df['pct_magnitude']   = df['title'].apply(extract_pct_magnitude)

# Composite keyword score
df['kw_net'] = (
    df['kw_strong_bull'] * 2 + df['kw_weak_bull']
  - df['kw_strong_bear'] * 2 - df['kw_weak_bear']
)

print("✅ Lexicon features computed.")
print()

# Validate: do keyword signals correlate with labels?
print("LEXICON SIGNAL VALIDATION (mean per label):")
kw_cols = ['kw_strong_bull','kw_strong_bear','kw_net',
           'kw_event','kw_analyst','has_price_target','pct_magnitude']
kw_by_label = df.groupby('label')[kw_cols].mean().round(3)
kw_by_label.index = [LABEL_MAP[i] for i in kw_by_label.index]
print(kw_by_label.to_string())
print()
print("💡 kw_net should be: SELL < 0 < BUY  (validates lexicon quality)")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Visual validation of keyword features vs labels
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 4, figsize=(22, 10))
axes = axes.flatten()

feature_configs = [
    ('kw_strong_bull', 'Strong Bullish Keywords'),
    ('kw_strong_bear', 'Strong Bearish Keywords'),
    ('kw_net',         'Net Keyword Score (bull-bear)'),
    ('kw_event',       'Event Keywords'),
    ('kw_analyst',     'Analyst Action Keywords'),
    ('has_price_target','Has Price Target (0/1)'),
    ('pct_magnitude',  'Pct Magnitude Mentioned'),
    ('token_count',    'Token Count (title length)'),
]

for ax, (col, title) in zip(axes, feature_configs):
    data_by_label = [df[df['label']==lv][col].values for lv in [0,1,2]]
    bp = ax.boxplot(data_by_label, patch_artist=True, notch=False,
                    showfliers=True, flierprops=dict(marker='.', alpha=0.3, ms=3))
    for patch, lv in zip(bp['boxes'], [0,1,2]):
        patch.set_facecolor(LABEL_PAL[lv])
        patch.set_alpha(0.75)
    ax.set_xticklabels([LABEL_MAP[lv] for lv in [0,1,2]])
    ax.set_title(title, fontweight='bold', fontsize=10)
    ax.set_ylabel("Value")

plt.suptitle("Keyword Feature Distributions by Label",
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig("step3_kw_validation.png", bbox_inches='tight', dpi=150)
plt.show()


## ⏱️ 8. Temporal & Structural Features

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Market timing features — WHEN the article was published matters because:
#   Pre-market articles (US: 4-9 UTC) → price hasn't reacted yet → strong signal
#   After-hours articles → may gap open next day → different reaction pattern
#   Weekend articles → no immediate price reaction possible
# ─────────────────────────────────────────────────────────────────────────────

df['pub_hour']     = df['pub_dt'].dt.hour
df['pub_dow']      = df['pub_dt'].dt.dayofweek       # 0=Mon
df['pub_month']    = df['pub_dt'].dt.month
df['pub_quarter']  = df['pub_dt'].dt.quarter

# Market session flags
df['is_us_premarket']   = ((df['pub_hour'] >= 4)  & (df['pub_hour'] < 9)  & (df['region']=='US')).astype(int)
df['is_us_market']      = ((df['pub_hour'] >= 14) & (df['pub_hour'] < 21) & (df['region']=='US')).astype(int)
df['is_in_premarket']   = ((df['pub_hour'] >= 0)  & (df['pub_hour'] < 4)  & (df['region']=='IN')).astype(int)
df['is_in_market']      = ((df['pub_hour'] >= 4)  & (df['pub_hour'] < 10) & (df['region']=='IN')).astype(int)
df['is_weekend']        = (df['pub_dow'] >= 5).astype(int)
df['is_friday_close']   = ((df['pub_dow'] == 4) & (df['pub_hour'] >= 18)).astype(int)

# Source credibility (binary from Step 1)
df['is_high_credibility'] = (df['source_credibility'] >= 0.75).astype(int)
df['is_india']            = (df['region'] == 'IN').astype(int)

# Rolling daily news volume per ticker
#   High volume = more information → higher signal reliability
vol_map = df.groupby(['ticker','pub_date']).size().to_dict()
df['daily_news_vol'] = df.apply(
    lambda r: vol_map.get((r['ticker'], r['pub_date']), 1), axis=1
)
df['log_news_vol'] = np.log1p(df['daily_news_vol'])

# Extreme move flag (captured in Stage 3 of cleaning)
# Already in df: flag_extreme_move, extreme_move_pct

print("Temporal & structural features added:")
new_cols = ['is_us_premarket','is_us_market','is_in_premarket','is_in_market',
            'is_weekend','is_friday_close','is_high_credibility','is_india',
            'daily_news_vol','log_news_vol']
print(df[new_cols].describe().round(3).to_string())


## 📋 9. Full Feature Inventory

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Complete list of features produced by Step 3
# (FinBERT features will be added in Step 4)
# ─────────────────────────────────────────────────────────────────────────────

NLP_FEATURES_STEP3 = [
    # Keyword lexicon
    'kw_strong_bull', 'kw_weak_bull', 'kw_strong_bear', 'kw_weak_bear',
    'kw_neutral', 'kw_event', 'kw_analyst', 'kw_india_bull', 'kw_india_bear',
    'kw_net', 'has_price_target', 'pct_magnitude',
    # Text structure
    'token_count', 'flag_extreme_move', 'extreme_move_pct',
    # Temporal
    'pub_hour', 'pub_dow', 'pub_month', 'pub_quarter',
    'is_us_premarket', 'is_us_market', 'is_in_premarket', 'is_in_market',
    'is_weekend', 'is_friday_close',
    # Source & market
    'is_high_credibility', 'is_india', 'daily_news_vol', 'log_news_vol',
]

print(f"Step 3 features  : {len(NLP_FEATURES_STEP3)}")
print(f"Step 4 will add  : ~7  (FinBERT pos/neg/neu/net/conf/entropy + rolling sentiment)")
print(f"Total expected   : ~{len(NLP_FEATURES_STEP3)+7}")
print()
print("FEATURE COMPLETENESS CHECK:")
for col in NLP_FEATURES_STEP3:
    nulls = df[col].isna().sum()
    status = "✅" if nulls == 0 else f"⚠️  {nulls} nulls"
    print(f"  {col:<28}: {status}")


In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Correlation of Step 3 features with label — quick predictive power check
# Before investing GPU time on FinBERT, confirm lexicon features are useful
# ─────────────────────────────────────────────────────────────────────────────
from scipy.stats import spearmanr

print("SPEARMAN CORRELATION WITH LABEL (absolute value):")
print("(higher = more individually predictive)")
print("="*50)

correlations = {}
for col in NLP_FEATURES_STEP3:
    try:
        r, p = spearmanr(df[col].fillna(0), df['label'])
        correlations[col] = (abs(r), p)
    except:
        correlations[col] = (0, 1)

corr_sorted = sorted(correlations.items(), key=lambda x: -x[1][0])
for feat, (r, p) in corr_sorted:
    sig = "***" if p < 0.001 else "** " if p < 0.01 else "*  " if p < 0.05 else "   "
    bar = '█' * int(r * 40)
    print(f"  {sig} {feat:<28} r={r:.4f}  {bar}")

from scipy.stats import spearmanr as _sr
print()
print("*** p<0.001   ** p<0.01   * p<0.05")


## 💾 10. Save Step 3 Output

In [ ]:
# Save everything Step 4 (FinBERT) needs
save_cols = (
    # Identifiers
    ['id','ticker','region','title','title_clean','published_at',
     'pub_date','pub_hour','pub_dow','pub_month','pub_quarter',
     # Labels
     'label','label_1d','label_2d','label_3d','horizons_agree',
     'ret_1d','ret_2d','ret_3d',
     # Source
     'source_name','source_credibility']
    + NLP_FEATURES_STEP3
)
save_cols = [c for c in save_cols if c in df.columns]

df[save_cols].to_csv("step3_nlp.csv", index=False)

print(f"✅ Saved: step3_nlp.csv — {len(df):,} rows, {len(save_cols)} columns")
print()

# ── Final report ──────────────────────────────────────────────────────────────
counts = df['label'].value_counts().sort_index()
total  = len(df)
imb    = counts.max() / counts.min()

print("╔══════════════════════════════════════════════════════════════════════╗")
print("║              STEP 3 SUMMARY — NLP PREPROCESSING DONE               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Articles (equity only)   : {total:>6,}                              ║")
print(f"║  Tickers                  : {df['ticker'].nunique():>6}                              ║")
print(f"║  Regions                  : US ({(df['region']=='US').sum():,}) + India ({(df['region']=='IN').sum():,})            ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
for lv in [0,1,2]:
    cnt = counts[lv]
    print(f"║  {LABEL_MAP[lv]:4s}               : {cnt:>5,}  ({cnt/total*100:5.1f}%)                      ║")
print(f"║  Imbalance ratio          : {imb:>5.2f}x                             ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print(f"║  Step 3 features built    : {len(NLP_FEATURES_STEP3):>6}                              ║")
print(f"║  Extreme move articles    : {df['flag_extreme_move'].sum():>6,} ({df['flag_extreme_move'].mean()*100:.1f}%)               ║")
print(f"║  High-confidence labels   : {(df['horizons_agree']==True).sum():>6,} ({(df['horizons_agree']==True).mean()*100:.1f}%)               ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  PLAN FOR STEP 4                                                    ║")
print("║  • Load ProsusAI/finbert (GPU recommended, T4 on Colab is fine)     ║")
print("║  • Batch inference on title_clean (batch_size=32)                   ║")
print("║  • Extract: pos/neg/neu scores, net sentiment, entropy, confidence  ║")
print("║  • Add rolling 3-article sentiment momentum per ticker              ║")
print("║  • Validate FinBERT features correlate with price labels            ║")
print("╠══════════════════════════════════════════════════════════════════════╣")
print("║  ✅  Review lift tokens & keyword boxplots before Step 4            ║")
print("╚══════════════════════════════════════════════════════════════════════╝")
